In [ ]:
# Figure 7: (a), (b), (c), (d), (e)

In [3]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import mean_squared_error

# ==========================================
# 1. USER CONFIGURATION & SETUP
# ==========================================
output_dir = r'C:\Users\PC1\Desktop\Code_Data\Data\Figure_7\output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Configurable start and end years for different analyses
start_year = 1985
end_year_corr = 2014  
end_year_bar = 2024   

# Set step-size for the Bar Chart X-axis
bar_x_tick_step = 3 

# Size & Spacing Configurations 
ind_figsize = (5.5, 3.5)              
bar_figsize = (7.0, 3.5)              
combined_figsize = (17.0, 8.5)        # Adjusted for 2 rows and 3 columns
label_fs = 14                         

# Panel label text and placement coordinates
panel_labels = {'bar': '(a)', 'ncrb': '(b)', 'imd': '(c)', 'loocv_n': '(d)', 'loocv_i': '(e)'}
panel_label_coords = {
    'bar': (0.01, 0.92), 
    'ncrb': (0.01, 0.90), 
    'imd': (0.008, 0.90),
    'loocv_n': (0.01, 0.90),
    'loocv_i': (0.01, 0.90)
}

# Output formats configuration
output_formats = ['png', 'svg']

# ==========================================
# 2. GLOBAL FONT & THEME SETTINGS
# ==========================================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'

custom_rc = {
    "font.family": "Times New Roman",
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.edgecolor": "black",
    "grid.color": "#c0c0c0"  
}
sns.set_theme(style="whitegrid", rc=custom_rc)

# Helper function to enforce thick borders, visible ticks, and bold tick labels
def polish_spines_and_ticks(ax, x_tick_step=None):
    # Force solid black spines
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1.0)
    
    # Force ticks to be visible
    ax.tick_params(axis='both', which='major', bottom=True, left=True, width=1.0, length=4, colors='black')
    
    # Apply step size for bar chart x-axis if provided
    if x_tick_step is not None:
        ax.set_xticks(ax.get_xticks()[::x_tick_step])
        
    # Force tick labels to be normal weight (not bold to prevent bleeding) and Times New Roman
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('normal')
        label.set_family('Times New Roman')
        label.set_fontsize('14')

# ==========================================
# 3. LOAD AND CLEAN DATA
# ==========================================
hi_file = r'C:\Users\PC1\Desktop\Code_Data\Data\Figure_7\Data\NCRB_IMD_Deaths_HI_Hazard_freq.xlsx'
death_file = r'C:\Users\PC1\Desktop\Code_Data\Data\Figure_7\Data\NCRB_IMD_Deaths_HI_Hazard_freq.xlsx'

df_hi = pd.read_excel(hi_file, sheet_name='Sheet2', header=1)
df_hist = df_hi[['Year', 'Danger', 'Extreme Danger', 'Danger+Extreme_Danger']].copy()
df_hist = df_hist.apply(pd.to_numeric, errors='coerce').dropna()

df_death = pd.read_excel(death_file)

# --- Data for Bar Chart ---
df_bar = df_death[(df_death['Year'] >= start_year) & (df_death['Year'] <= end_year_bar)].copy()
df_melt = df_bar.melt(id_vars=['Year'], value_vars=['NCRB', 'IMD'], var_name='Source', value_name='Deaths')

# --- Data for Correlation Plots ---
df_merged = pd.merge(df_hist, df_death, on='Year', how='inner')
df_corr = df_merged[(df_merged['Year'] >= start_year) & (df_merged['Year'] <= end_year_corr)].copy()

# Separate clean subsets for Correlation
df_ncrb = df_corr.dropna(subset=['NCRB', 'Danger+Extreme_Danger'])
X_n = df_ncrb['Danger+Extreme_Danger']
Y_n = df_ncrb['NCRB']

df_imd = df_corr.dropna(subset=['IMD', 'Danger+Extreme_Danger'])
X_i = df_imd['Danger+Extreme_Danger']
Y_i = df_imd['IMD']

# ==========================================
# 4. PERFORM LINEAR REGRESSIONS & LOOCV 
# ==========================================
# Base Linear Regressions
slope_n, intercept_n, r_val_n, p_val_n, _ = stats.linregress(X_n, Y_n)
r2_n = r_val_n ** 2
fit_label_n = f'$Y = {slope_n:.2f}X {intercept_n:+.2f},$ $R^2 = {r2_n:.2f}, p = {p_val_n:.4f}$'

slope_i, intercept_i, r_val_i, p_val_i, _ = stats.linregress(X_i, Y_i)
r2_i = r_val_i ** 2
fit_label_i = f'$Y = {slope_i:.2f}X {intercept_i:+.2f},$ $R^2 = {r2_i:.2f}, p = {p_val_i:.4f}$'

# LOOCV Function
def calculate_loocv(X_series, Y_series):
    X = X_series.values
    Y = Y_series.values
    preds = np.zeros(len(X))
    for i in range(len(X)):
        X_train = np.delete(X, i)
        Y_train = np.delete(Y, i)
        slope, intercept, _, _, _ = stats.linregress(X_train, Y_train)
        preds[i] = slope * X[i] + intercept
    return Y, preds

# Calculate LOOCV Predictions, RMSE, and Min/Max Bounds
actual_n, pred_n = calculate_loocv(X_n, Y_n)
rmse_loocv_n = np.sqrt(mean_squared_error(actual_n, pred_n))
min_val_n = min(actual_n.min(), pred_n.min()) - 100
max_val_n = max(actual_n.max(), pred_n.max()) + 100

actual_i, pred_i = calculate_loocv(X_i, Y_i)
rmse_loocv_i = np.sqrt(mean_squared_error(actual_i, pred_i))
min_val_i = min(actual_i.min(), pred_i.min()) - 100
max_val_i = max(actual_i.max(), pred_i.max()) + 100

# ==========================================
# 5. GENERATE INDIVIDUAL PLOTS (RESIDUALS)
# ==========================================

# NCRB Residuals
fig, ax = plt.subplots(figsize=ind_figsize)
res_n = Y_n - (slope_n * X_n + intercept_n)
sns.scatterplot(x=X_n, y=res_n, ax=ax, s=25, color='#002244', alpha=0.95, edgecolor='black', linewidth=0.8)
ax.axhline(0, color='#cc0000', linestyle='--', linewidth=2)
ax.set_xlabel("Frequency of Heat Hazard Days", fontsize=11, weight='bold', color='black')
ax.set_ylabel("Residuals (NCRB)", fontsize=11, weight='bold', color='black')
polish_spines_and_ticks(ax)
plt.tight_layout()
for ext in output_formats:
    plt.savefig(os.path.join(output_dir, f'Residual_Scatter_NCRB.{ext}'), dpi=600, bbox_inches='tight')
plt.close()

# IMD Residuals
fig, ax = plt.subplots(figsize=ind_figsize)
res_i = Y_i - (slope_i * X_i + intercept_i)
sns.scatterplot(x=X_i, y=res_i, ax=ax, s=25, color='#002244', alpha=0.95, edgecolor='black', linewidth=0.8)
ax.axhline(0, color='#cc0000', linestyle='--', linewidth=2)
ax.set_xlabel("Frequency of Heat Hazard Days", fontsize=11, weight='bold', color='black')
ax.set_ylabel("Residuals (IMD)", fontsize=11, weight='bold', color='black')
polish_spines_and_ticks(ax)
plt.tight_layout()
for ext in output_formats:
    plt.savefig(os.path.join(output_dir, f'Residual_Scatter_IMD.{ext}'), dpi=600, bbox_inches='tight')
plt.close()


# ==========================================
# 6. GENERATE COMBINED MULTI-PANEL FIGURE (5 Panels - 2 Rows)
# ==========================================
fig = plt.figure(figsize=combined_figsize)
gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.0], width_ratios=[1.0, 1.0, 1.0], hspace=0.25, wspace=0.28)

# --- ROW 1 ---
# Panel (a): Bar chart spanning the first two columns (0 and 1)
ax_bar = fig.add_subplot(gs[0, 0:2])
# Panel (d): NCRB LOOCV next to the bar chart in column 2
ax_loocv_n = fig.add_subplot(gs[0, 2])

# --- ROW 2 ---
# Panel (b): NCRB Corr in column 0
ax_ncrb = fig.add_subplot(gs[1, 0])
# Panel (c): IMD Corr in column 1
ax_imd = fig.add_subplot(gs[1, 1])
# Panel (e): IMD LOOCV in column 2 (directly below 'd' and next to 'c')
ax_loocv_i = fig.add_subplot(gs[1, 2])

# ------------------------------------------
# Panel (a): Bar Plot
# ------------------------------------------
sns.barplot(data=df_melt, x='Year', y='Deaths', hue='Source', palette=['#FF0000', '#FFB3DD'], ax=ax_bar, edgecolor='black', linewidth=0.4)
ax_bar.set_xlabel("") 
ax_bar.set_ylabel("Heat-related Mortality", fontsize=16, weight='bold', color='black')                                  
ax_bar.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': 12}, frameon=True, edgecolor='black')
ax_bar.tick_params(axis='x', rotation=0)
polish_spines_and_ticks(ax_bar, x_tick_step=bar_x_tick_step)
ax_bar.text(panel_label_coords['bar'][0], panel_label_coords['bar'][1], panel_labels['bar'], transform=ax_bar.transAxes,
            fontsize=label_fs, weight='bold', family='Times New Roman', color='black')

# ------------------------------------------
# Panel (b): NCRB Correlation Plot
# ------------------------------------------
sns.regplot(x=X_n, y=Y_n, ax=ax_ncrb,
            scatter_kws={'s': 20, 'color': '#002244', 'alpha': 0.95, 'edgecolors': 'black', 'linewidths': 0.4},
            line_kws={'color': '#cc0000', 'linewidth': 1.8, 'label': fit_label_n})
ax_ncrb.set_xlabel("Frequency of Heat Hazard Days", fontsize=16, weight='bold', color='black')
ax_ncrb.set_ylabel("Heat-related Mortality (NCRB)", fontsize=16, weight='bold', color='black')
ax_ncrb.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': 12}, frameon=True, edgecolor='black')
polish_spines_and_ticks(ax_ncrb)
ax_ncrb.text(panel_label_coords['ncrb'][0], panel_label_coords['ncrb'][1], panel_labels['ncrb'], transform=ax_ncrb.transAxes,
             fontsize=label_fs, weight='bold', family='Times New Roman', color='black')

# ------------------------------------------
# Panel (c): IMD Correlation Plot
# ------------------------------------------
sns.regplot(x=X_i, y=Y_i, ax=ax_imd,
            scatter_kws={'s': 20, 'color': '#002244', 'alpha': 0.95, 'edgecolors': 'black', 'linewidths': 0.4},
            line_kws={'color': '#cc0000', 'linewidth': 1.8, 'label': fit_label_i})
ax_imd.set_xlabel("Frequency of Heat Hazard Days", fontsize=16, weight='bold', color='black')
ax_imd.set_ylabel("Heat-related Mortality (IMD)", fontsize=16, weight='bold', color='black')
ax_imd.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': 12}, frameon=True, edgecolor='black')
polish_spines_and_ticks(ax_imd)
ax_imd.text(panel_label_coords['imd'][0], panel_label_coords['imd'][1], panel_labels['imd'], transform=ax_imd.transAxes,
            fontsize=label_fs, weight='bold', family='Times New Roman', color='black')

# ------------------------------------------
# Panel (d): NCRB LOOCV Scatter Plot
# ------------------------------------------
ax_loocv_n.scatter(actual_n, pred_n, s=20, color='#002244', alpha=0.95, edgecolors='black', linewidths=0.4, label='LOOCV Data')
ax_loocv_n.plot([min_val_n, max_val_n], [min_val_n, max_val_n], 'r--', lw=1.8, label='1:1 Perfect Fit')
ax_loocv_n.set_xlabel("Observed Mortality (NCRB)", fontsize=16, weight='bold', color='black')
ax_loocv_n.set_ylabel("Predicted Mortality (NCRB)", fontsize=16, weight='bold', color='black')
props = dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9, edgecolor='black', linewidth=0.8)
ax_loocv_n.text(0.12, 0.95, f'RMSE = {rmse_loocv_n:.0f}', transform=ax_loocv_n.transAxes, 
                fontsize=12, weight='bold', family='Times New Roman', verticalalignment='top', bbox=props)
ax_loocv_n.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': 12}, frameon=True, edgecolor='black')
polish_spines_and_ticks(ax_loocv_n)
ax_loocv_n.text(panel_label_coords['loocv_n'][0], panel_label_coords['loocv_n'][1], panel_labels['loocv_n'], transform=ax_loocv_n.transAxes,
             fontsize=label_fs, weight='bold', family='Times New Roman', color='black')

# ------------------------------------------
# Panel (e): IMD LOOCV Scatter Plot
# ------------------------------------------
ax_loocv_i.scatter(actual_i, pred_i, s=20, color='#002244', alpha=0.95, edgecolors='black', linewidths=0.4, label='LOOCV Data')
ax_loocv_i.plot([min_val_i, max_val_i], [min_val_i, max_val_i], 'r--', lw=1.8, label='1:1 Perfect Fit')
ax_loocv_i.set_xlabel("Observed Mortality (IMD)", fontsize=16, weight='bold', color='black')
ax_loocv_i.set_ylabel("Predicted Mortality (IMD)", fontsize=16, weight='bold', color='black')
ax_loocv_i.text(0.12, 0.95, f'RMSE = {rmse_loocv_i:.0f}', transform=ax_loocv_i.transAxes, 
                fontsize=12, weight='bold', family='Times New Roman', verticalalignment='top', bbox=props)
ax_loocv_i.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': 12}, frameon=True, edgecolor='black')
polish_spines_and_ticks(ax_loocv_i)
ax_loocv_i.text(panel_label_coords['loocv_i'][0], panel_label_coords['loocv_i'][1], panel_labels['loocv_i'], transform=ax_loocv_i.transAxes,
             fontsize=label_fs, weight='bold', family='Times New Roman', color='black')

# ------------------------------------------
# Final Export
# ------------------------------------------
for ext in output_formats:
    plt.savefig(os.path.join(output_dir, f'Figure_7.{ext}'), dpi=1200, bbox_inches='tight')
plt.close()

print(f"\nAll plots successfully generated and saved (.png, .svg, .jpg, .jpeg, .pdf) to: {output_dir}")


All plots successfully generated and saved (.png, .svg, .jpg, .jpeg, .pdf) to: C:\Users\PC1\Desktop\Code_Data\Data\Figure_7\output
